## Texas and Georgia Auction.com

In [487]:
import pandas as pd
from pathlib import Path
import numpy as np
from rapidfuzz import process, fuzz
import subprocess
import json
import os

In [488]:
folder = Path('raw_data/auction_com')

excel_files = list(folder.glob('*.xlsx'))

# Combine all texas
auction = pd.concat(
    [pd.read_excel(file, dtype={'Tax ID': 'string'}) for file in excel_files],
    ignore_index=True
)

/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/Caskroom/miniconda/base/envs/py38/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style,

In [489]:
print(f"Files combined: {len(excel_files)}")
print(f"Total rows: {len(auction)}")
print(f"Total columns: {len(auction.columns)}")

display(auction.head())

Files combined: 4
Total rows: 1946
Total columns: 19


,Listing Type,Property Address,Property City,County,State,Zip,Status,Auction Date,Property Type,Square Footage,Bedrooms,Baths,Lot Size,Year Built,Est. Market Value,Starting Bid,Est. Credit Bid,APN,Trustee Number
0,Foreclosure,1827 POSEIDON DRIVE,LANCASTER,Dallas,TX,75134.0,SALE_PENDING,2026-11-03,Single family home,2599.0,4.0,3.0,0.15,2006.0,327065.0,NaN,NaN,NaN,TS 10950319
1,Foreclosure,309 ADAMTOWNE DR,EL PASO,El Paso,TX,79928.0,SALE_PENDING,2026-11-03,Single family home,1932.0,4.0,3.0,0.12,2024.0,313422.0,NaN,NaN,NaN,TS 10936573
2,Foreclosure,3011 LAYNE ST,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,1322.0,3.0,2.0,0.11,2022.0,230043.0,NaN,NaN,NaN,26-000441-671
3,Foreclosure,305 HAYWOOD DR,BENBROOK,Tarrant,TX,76126.0,SALE_PENDING,2026-11-03,Single family home,1911.0,4.0,2.0,0.22,1980.0,334986.0,NaN,NaN,NaN,9926-3917
4,Foreclosure,204 KING WILLIAM DR,LA PORTE,Harris,TX,77571.0,SALE_PENDING,2026-11-03,Single family home,3341.0,4.0,2.5,0.15,2006.0,308511.0,NaN,NaN,126-056-001-0005,26-013461


In [490]:
#Create full address for record name

auction["Last Name"] = (
    auction["Property Address"].str.title() + ", " +
    auction["Property City"].str.title() + ", " +
    auction["State"].str.title() + " " +
    auction["Zip"].astype(str).str.replace(r"\.0$", "", regex=True)
)

## Match Addresses

In [491]:
query = """
SELECT
    Id,
    Name,
    Auction_Status__c,
    Auction_Trustee_Lookup__r.Name,
    Auction_Listing_ID__c,
    Auction_Opening_Bid__c,
    Online_Bidding__c
FROM Lead
WHERE (
    Auction_Date__c = LAST_N_DAYS:60
    OR Auction_Date__c = NEXT_N_DAYS:30)
AND State_Lookup__r.Name IN ('Texas', 'Georgia')
"""

result = subprocess.run(
    [
        "sf", "data", "query",
        "--query", query,
        "--target-org", "production_org",
        "--json"
    ],
    capture_output=True,
    text=True,
    env={**os.environ, "NO_COLOR": "1", "FORCE_COLOR": "0"}
)

if result.returncode != 0:
    print("Salesforce Error:")
    print(result.stderr)
    print(result.stdout[:1000])

else:
    # Remove ANSI color codes if any remain
    import re

    clean_output = re.sub(
        r'\x1b\[[0-9;]*m',
        '',
        result.stdout
    )

    data = json.loads(clean_output)

    sf_leads = pd.json_normalize(data['result']['records'])

    print(f"Successfully retrieved {len(sf_leads)} Salesforce records")
    display(sf_leads.head())

Successfully retrieved 5799 Salesforce records


,Id,Name,Auction_Status__c,Auction_Listing_ID__c,Auction_Opening_Bid__c,Online_Bidding__c,attributes.type,attributes.url,Auction_Trustee_Lookup__r.attributes.type,Auction_Trustee_Lookup__r.attributes.url,Auction_Trustee_Lookup__r.Name
0,00QQg00000nIBX3MAO,"209 Rosewood Drive, Brunswick, GA 31525",Killed,2183399,100755.64,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
1,00QQg00000nIBX4MAO,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,2182824,161200.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
2,00QQg00000nIBX5MAO,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,2182090,224360.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
3,00QQg00000nIBX6MAO,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,2181984,NaN,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com
4,00QQg00000nIBX7MAO,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,2181983,83840.00,None,Lead,/services/data/v68.0/sobjects/Lead/00QQg00000n...,Contact__c,/services/data/v68.0/sobjects/Contact__c/a1fQg...,Auction.com


In [492]:
#Rename sf columns 

sf_leads = sf_leads.rename(columns={
    'Name': 'Last Name',
    'Auction_Status__c': 'Auction Status',
    'Auction_Trustee_Lookup__r.Name': 'Auction Trustee',
    'Auction_Listing_ID__c': 'Auction Listing ID',
    'Auction_Opening_Bid__c': 'Auction Opening Bid',
    'Online_Bidding__c': "Online Bidding"
})

In [493]:
sf_leads[['Last Name', 'Auction Status', 'Auction Trustee',
          'Auction Listing ID', 'Auction Opening Bid']].head()

,Last Name,Auction Status,Auction Trustee,Auction Listing ID,Auction Opening Bid
0,"209 Rosewood Drive, Brunswick, GA 31525",Killed,Auction.com,2183399,100755.64
1,"1604 Cave Spring Rd Sw, Rome, GA 30161",Killed,Auction.com,2182824,161200.00
2,"6851 Alexander Pkwy, Douglasville, GA 30135",Killed,Auction.com,2182090,224360.00
3,"4873 Old Belair Lane, Grovetown, GA 30813",Killed,Auction.com,2181984,NaN
4,"10475 Ga Highway 57, Mc Intyre, GA 31054",Killed,Auction.com,2181983,83840.00


In [494]:
#Fuzzy Match

threshold = 85

# Normalize names
def normalize_name(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).upper().strip().split())

# Salesforce lookup
sf_lookup = {}

for name in sf_leads['Last Name'].dropna().unique():
    normalized = normalize_name(name)
    if normalized:
        sf_lookup.setdefault(normalized, name)

sf_names = list(sf_lookup.keys())

# Keep original Auction.com Last Name
auction['Auction Last Name'] = auction['Last Name']

# Fuzzy match
def fuzzy_match_name(name):
    normalized = normalize_name(name)

    if not normalized or not sf_names:
        return pd.Series([None, None])

    match = process.extractOne(
        normalized,
        sf_names,
        scorer=fuzz.ratio
    )

    if match:
        matched_name, score, _ = match
        return pd.Series([sf_lookup[matched_name], round(score, 2)])

    return pd.Series([None, None])

# Store Salesforce match and confidence
auction[['Salesforce Last Name', 'Match Confidence']] = (
    auction['Auction Last Name'].apply(fuzzy_match_name)
)

# Use Salesforce Last Name only if confidence >= 85
auction['Last Name'] = auction['Auction Last Name']

matched = auction['Match Confidence'] >= threshold

auction.loc[matched, 'Last Name'] = (
    auction.loc[matched, 'Salesforce Last Name']
)

In [495]:
#View matches

display(auction[
    ['Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Auction Last Name,Salesforce Last Name,Match Confidence
0,"1827 Poseidon Drive, Lancaster, Tx 75134","927 Poinsettia Drive, Lancaster, TX 75146",83.95
1,"309 Adamtowne Dr, El Paso, Tx 79928","12238 Rathmore Dr, El Paso, TX 79928",78.87
2,"3011 Layne St, La Porte, Tx 77571","3135 Hamilton St, La Porte, TX 77571",81.16
3,"305 Haywood Dr, Benbrook, Tx 76126","7305 Hogan Dr, Benbrook, TX 76126",86.57
4,"204 King William Dr, La Porte, Tx 77571","704 Garden Walk Dr, La Porte, TX 77571",77.92
...,...,...,...
1941,"577 Everett Springs Rd Sw, Calhoun, Ga 30701","1604 Cave Spring Rd Sw, Rome, GA 30161",65.85
1942,"3916 Duchess Ct, Valdosta, Ga 31605","5003 Peachtree St, Valdosta, GA 31605",75.00
1943,"1621 Alcovy Ridge Crossing, Loganville, Ga 30052","702 Buckhorn Ct, Loganville, GA 30052",68.24
1944,NaN,None,NaN


In [496]:
#Look at differences

#Common street suffixes
suffix_map = {
    'DRIVE': 'DR',
    'STREET': 'ST',
    'AVENUE': 'AVE',
    'ROAD': 'RD',
    'LANE': 'LN',
    'BOULEVARD': 'BLVD',
    'COURT': 'CT',
    'CIRCLE': 'CIR',
    'PLACE': 'PL',
    'TERRACE': 'TER',
    'TRAIL': 'TRL',
    'PARKWAY': 'PKWY',
    'HIGHWAY': 'HWY',
    'WAY': 'WAY'
}

def get_address_parts(address):
    if pd.isna(address):
        return '', ''

    address = str(address).strip()
    parts = [p.strip() for p in address.split(',')]

    # Expected format: Street, City, TX ZIP
    street = parts[0] if len(parts) >= 1 else ''
    city = parts[1] if len(parts) >= 3 else ''

    return street.upper(), city.upper()

def normalize_street(street):
    street = re.sub(r'[^\w\s]', '', street.upper())
    words = street.split()
    return ' '.join(suffix_map.get(word, word) for word in words)

def compare_addresses(row):
    auction_street, auction_city = get_address_parts(row['Auction Last Name'])
    sf_street, sf_city = get_address_parts(row['Salesforce Last Name'])

    # Detect abbreviation-only differences
    suffix_difference = (
        auction_street != sf_street
        and normalize_street(auction_street) == normalize_street(sf_street)
    )

    city_difference = auction_city != sf_city

    return pd.Series({
        'Street Suffix Difference': suffix_difference,
        'City Difference': city_difference,
        'Auction City': auction_city,
        'Salesforce City': sf_city
    })

# Compare matched records
address_comparison = auction[
    (auction['Match Confidence'] >= 85) &
    auction['Salesforce Last Name'].notna()
].copy()

address_comparison = address_comparison.join(
    address_comparison.apply(compare_addresses, axis=1)
)

In [497]:
differences = address_comparison[
    address_comparison['Street Suffix Difference'] |
    address_comparison['City Difference']
]

display(differences[
    [
        'Last Name',
        'Auction Last Name',
        'Salesforce Last Name',
        'Match Confidence',
        'Street Suffix Difference',
        'City Difference'
    ]
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence,Street Suffix Difference,City Difference
126,"9450 Valley Way Dr, San Antonio, TX 78250","9450 Valley Way Drive, San Antonio, Tx 78250","9450 Valley Way Dr, San Antonio, TX 78250",96.47,True,False
187,"612 Bunker Hill Rd, Fate, TX 75189","612 Bunker Hill Road, Fate, Tx 75189","612 Bunker Hill Rd, Fate, TX 75189",97.14,True,False
382,"9607 Walnut St, Dallas, TX 75243","9607 Walnut Street, Dallas, Tx 75243","9607 Walnut St, Dallas, TX 75243",94.12,True,False
550,"7806 Garner Rd, Rowlett, TX 75088","7806 Garner Road, Rowlett, Tx 75088","7806 Garner Rd, Rowlett, TX 75088",97.06,True,False
564,"12610 Raven South Dr, Cypress, TX 77429","12610 Raven South Drive, Cypress, Tx 77429","12610 Raven South Dr, Cypress, TX 77429",96.30,True,False
646,"11720 Kurth Dr, Frisco, TX 75036","11720 Kurth Drive, Frisco, Tx 75034","11720 Kurth Dr, Frisco, TX 75036",92.54,True,False
648,"6742 Spring Hurst St, San Antonio, TX 78249","6742 Spring Hurst Street, San Antonio, Tx 78249","6742 Spring Hurst St, San Antonio, TX 78249",95.56,True,False
734,"1302 Harbor Ct, Lancaster, TX 75134","1302 Harbor Court, Lancaster, Tx 75134","1302 Harbor Ct, Lancaster, TX 75134",95.89,True,False
777,"9643 Towne Lake Pkwy, Cypress, TX 77433","9643 Towne Lake Parkway, Cypress, Tx 77433","9643 Towne Lake Pkwy, Cypress, TX 77433",96.30,True,False
781,"13222 Arden Ridge Ln, Houston, TX 77014","13222 Arden Ridge Lane, Houston, Tx 77014","13222 Arden Ridge Ln, Houston, TX 77014",97.50,True,False


In [498]:
print("Street suffix differences:",
      address_comparison['Street Suffix Difference'].sum())

print("City differences:",
      address_comparison['City Difference'].sum())

print("Both:",
      (
          address_comparison['Street Suffix Difference'] &
          address_comparison['City Difference']
      ).sum())

Street suffix differences: 18
City differences: 0
Both: 0


In [499]:
low_confidence = auction[
    (auction['Match Confidence'] < 85) |
    (auction['Match Confidence'].isna())
]

display(low_confidence[
    ['Last Name','Auction Last Name', 'Salesforce Last Name', 'Match Confidence']
])

,Last Name,Auction Last Name,Salesforce Last Name,Match Confidence
0,"1827 Poseidon Drive, Lancaster, Tx 75134","1827 Poseidon Drive, Lancaster, Tx 75134","927 Poinsettia Drive, Lancaster, TX 75146",83.95
1,"309 Adamtowne Dr, El Paso, Tx 79928","309 Adamtowne Dr, El Paso, Tx 79928","12238 Rathmore Dr, El Paso, TX 79928",78.87
2,"3011 Layne St, La Porte, Tx 77571","3011 Layne St, La Porte, Tx 77571","3135 Hamilton St, La Porte, TX 77571",81.16
4,"204 King William Dr, La Porte, Tx 77571","204 King William Dr, La Porte, Tx 77571","704 Garden Walk Dr, La Porte, TX 77571",77.92
5,"2732 Trophy Club Dr, Trophy Club, Tx 76262","2732 Trophy Club Dr, Trophy Club, Tx 76262","2823 Macquarie, Trophy Club, TX 76262",73.42
...,...,...,...,...
1941,"577 Everett Springs Rd Sw, Calhoun, Ga 30701","577 Everett Springs Rd Sw, Calhoun, Ga 30701","1604 Cave Spring Rd Sw, Rome, GA 30161",65.85
1942,"3916 Duchess Ct, Valdosta, Ga 31605","3916 Duchess Ct, Valdosta, Ga 31605","5003 Peachtree St, Valdosta, GA 31605",75.00
1943,"1621 Alcovy Ridge Crossing, Loganville, Ga 30052","1621 Alcovy Ridge Crossing, Loganville, Ga 30052","702 Buckhorn Ct, Loganville, GA 30052",68.24
1944,NaN,NaN,None,NaN


## Format Data

In [500]:
#Online Bidding
auction['Online Bidding'] = 'Yes'

#Auction Trustee
auction['Auction Trustee (Lookup)'] = 'Auction.com'

In [501]:
# Remove blank auction rows
before = len(auction)

auction = auction.dropna(
    subset=['Last Name', 'State', 'oldCounty'],
    how='all'
).copy()

removed = before - len(auction)

print("Total auctions removed:", removed)
print("Total auctions remaining:", len(auction))

KeyError: ['oldCounty']

In [ ]:

texascounties = pd.read_csv('sf_data/texascounties.csv')
georgiacounties = pd.read_csv('sf_data/georgiacounties.csv')

# Rename original County
if 'oldCounty' not in auction.columns:
    auction = auction.rename(columns={'County': 'oldCounty'})

# DeKalb exception
auction.loc[
    auction['oldCounty']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.upper()
    .eq('DEKALB'),
    'oldCounty'
] = 'Dekalb'

# Normalize county names for matching only
def normalize_county(value):
    if pd.isna(value):
        return ''
    return str(value).strip().upper()

# Create county lookups
county_lookup = {}

for state, df in [('TX', texascounties), ('GA', georgiacounties)]:
    for _, row in df.iterrows():
        county = normalize_county(row['County'])

        county_lookup[(state, county)] = row['Market']

# Find Market
def find_county_details(row):

    if pd.isna(row['State']) or pd.isna(row['oldCounty']):
        return ''

    state = str(row['State']).strip().upper()
    county = normalize_county(row['oldCounty'])

    return county_lookup.get((state, county), '')

# Add Market to auction
auction['Market'] = auction.apply(
    find_county_details, axis=1
)

# Create County column
auction['County'] = (
    auction['oldCounty'].astype('string') +
    ' - ' +
    auction['State'].astype('string')
)

# Create Decision column
auction['Decision'] = 'Add New'

auction.loc[
    auction['Salesforce Last Name'].notna() &
    auction['Salesforce Last Name'].astype(str).str.strip().ne(''),
    'Decision'
] = 'Update'

auction.loc[
    auction['Market'] == 'Exclude',
    'Decision'
] = 'Exclude'

# Check for missing Markets
missing_markets = auction[
    auction['Market'].isna() |
    auction['Market'].astype(str).str.strip().eq('')
]

print("Total auctions:", len(auction))
print("Missing Markets:", len(missing_markets))
print("Market column exists:", 'Market' in auction.columns)

display(missing_markets[
    ['Last Name', 'State', 'oldCounty', 'Market']
])


Total auctions: 1946
Missing Markets: 8
Market column exists: True


,Last Name,State,oldCounty,Market
1184,NaN,NaN,NaN,
1185,NaN,NaN,NaN,
1296,NaN,NaN,NaN,
1297,NaN,NaN,NaN,
1319,NaN,NaN,NaN,
1320,NaN,NaN,NaN,
1944,NaN,NaN,NaN,
1945,NaN,NaN,NaN,


In [ ]:
print("auction:", 'Market' in auction.columns)
print("texascounties:", 'Market' in texascounties.columns)
print("georgiacounties:", 'Market' in georgiacounties.columns)

auction: True
texascounties: True
georgiacounties: True


## Lead Manager

In [ ]:
# # Create Salesforce lookup
# sf_lookup = (
#     sf_leads.drop_duplicates(subset='Last Name')
#     .set_index('Last Name')
# )

# # Get Salesforce values from fuzzy match
# auction['Salesforce Auction Trustee'] = (
#     auction['Salesforce Last Name']
#     .map(sf_lookup['Auction Trustee'])
# )

# auction['Salesforce Online Bidding'] = (
#     auction['Salesforce Last Name']
#     .map(sf_lookup['Online Bidding'])
# )

# # Create county manager lookups
# manager_county_lookup = {}

# for _, row in texascounties.iterrows():
#     manager_county_lookup[('TX', row['County'])] = {
#         'Online': row['Online Lead Manager'],
#         'In Person': row['In Person Lead Manager']
#     }

# for _, row in georgiacounties.iterrows():
#     manager_county_lookup[('GA', row['County'])] = {
#         'Lead Manager': row['Lead Manager']
#     }

# # Determine Lead Manager
# def find_lead_manager(row):
#     details = manager_county_lookup.get(
#         (row['State'], row['Normalized County']), {}
#     )

#     if row['State'] == 'TX':

#         use_online = (
#             str(row['Salesforce Auction Trustee']).strip() == 'Auction.com'
#             or
#             str(row['Salesforce Online Bidding']).strip() == 'Yes'
#         )

#         if use_online:
#             return details.get('Online', '')
#         else:
#             return details.get('In Person', '')

#     elif row['State'] == 'GA':
#         return details.get('Lead Manager', '')

#     return ''

# # Assign Lead Manager Name
# auction['Lead Manager Name'] = auction.apply(
#     find_lead_manager, axis=1
# )

# # Find Salesforce ID
# leadmanagers = pd.read_csv(
#     'sf_data/leadmanagers.csv',
#     dtype=str
# )

# manager_lookup = (
#     leadmanagers.drop_duplicates(subset='Lead Manager')
#     .set_index('Lead Manager')['Salesforce ID']
# )

# auction['Lead Manager'] = (
#     auction['Lead Manager Name'].map(manager_lookup)
# )

In [ ]:
# FIRST IMPORT ONLY

# Normalize county names for matching
def normalize_manager_county(value):
    if pd.isna(value):
        return ''

    county = str(value).strip().upper()

    # DeKalb exception
    if county.replace(' ', '') == 'DEKALB':
        return 'DEKALB'

    return county


# Create county manager lookup
manager_county_lookup = {}

for _, row in texascounties.iterrows():
    manager_county_lookup[
        ('TX', normalize_manager_county(row['County']))
    ] = row['Online Lead Manager']

for _, row in georgiacounties.iterrows():
    manager_county_lookup[
        ('GA', normalize_manager_county(row['County']))
    ] = row['Lead Manager Name']


# Find Lead Manager Name
def find_lead_manager(row):

    # Do not assign manager if Market = Exclude
    if str(row['Market']).strip().lower() == 'exclude':
        return ''

    if pd.isna(row['State']) or pd.isna(row['oldCounty']):
        return ''

    state = str(row['State']).strip().upper()
    county = normalize_manager_county(row['oldCounty'])

    manager = manager_county_lookup.get((state, county), '')

    # Return blank instead of NaN
    if pd.isna(manager):
        return ''

    return str(manager).strip()


# Assign Lead Manager Name
auction['Lead Manager Name'] = auction.apply(
    find_lead_manager, axis=1
)


# Find Lead Manager Salesforce ID
leadmanagers = pd.read_csv(
    'sf_data/leadmanagers.csv',
    dtype=str
)

# Clean manager names for matching
leadmanagers['Lead Manager'] = (
    leadmanagers['Lead Manager'].str.strip()
)

manager_lookup = (
    leadmanagers.drop_duplicates(subset='Lead Manager')
    .set_index('Lead Manager')['Salesforce ID']
)

auction['Lead Manager'] = (
    auction['Lead Manager Name'].map(manager_lookup)
)


# Keep excluded markets' Lead Manager blank
excluded = (
    auction['Market']
    .astype('string')
    .str.strip()
    .str.lower()
    .eq('exclude')
    .fillna(False)
)

auction.loc[
    excluded,
    ['Lead Manager Name', 'Lead Manager']
] = ''


# Find auctions missing a Lead Manager
missing_managers = auction[
    ~excluded &
    (
        auction['Lead Manager Name'].isna() |
        auction['Lead Manager Name'].eq('') |
        auction['Lead Manager'].isna() |
        auction['Lead Manager'].eq('')
    )
]


# Results
print("Total auctions:", len(auction))
print("Excluded Markets:", excluded.sum())
print("Missing Lead Managers:", len(missing_managers))

display(missing_managers[
    ['Last Name', 'State', 'oldCounty', 'Market',
     'Lead Manager Name', 'Lead Manager']
])


Total auctions: 1946
Excluded Markets: 419
Missing Lead Managers: 8


,Last Name,State,oldCounty,Market,Lead Manager Name,Lead Manager
1184,NaN,NaN,NaN,,,NaN
1185,NaN,NaN,NaN,,,NaN
1296,NaN,NaN,NaN,,,NaN
1297,NaN,NaN,NaN,,,NaN
1319,NaN,NaN,NaN,,,NaN
1320,NaN,NaN,NaN,,,NaN
1944,NaN,NaN,NaN,,,NaN
1945,NaN,NaN,NaN,,,NaN


In [ ]:
display(auction[
    [
        'Last Name',
        'Salesforce Last Name',
        'Salesforce Auction Trustee',
        'Salesforce Online Bidding',
        'Lead Manager Name',
        'Lead Manager'
    ]
].head(20))

KeyError: "['Salesforce Auction Trustee', 'Salesforce Online Bidding'] not in index"

In [ ]:
#No lead manager

display(auction[
    auction['Lead Manager Name'].isna() |
    (auction['Lead Manager Name'].astype(str).str.strip() == '')
])

,Listing Type,Property Address,Property City,oldCounty,State,Zip,Status,Auction Date,Property Type,Square Footage,...,Match Confidence,Online Bidding,Auction Trustee (Lookup),Market,County,Decision,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager
8,Foreclosure,7031 KRAUSE DR,MISSOURI CITY,Fort Bend,TX,77489.0,SALE_PENDING,2026-11-03,Single family home,1444.0,...,64.00,Yes,Auction.com,Exclude,Fort Bend - TX,Exclude,Auction.com,Yes,NaN,NaN
32,Foreclosure,14869 PRICKLY PEAR COVE,MAGNOLIA,Montgomery,TX,77357.0,SALE_PENDING,2026-11-03,Single family home,1339.0,...,69.05,Yes,Auction.com,Exclude,Montgomery - TX,Exclude,Auction.com,Yes,NaN,NaN
46,Foreclosure,2508 GOODNIGHT TRAIL,MANSFIELD,TARRANT,TX,76063.0,SALE_PENDING,2026-11-03,Single family home,2956.0,...,76.54,Yes,Auction.com,,TARRANT - TX,Update,Lookup Needed,None,,NaN
68,Foreclosure,22355 DOMINGUEZ DRIVE,PORTER,Montgomery,TX,77365.0,SALE_PENDING,2026-11-03,Single family home,2310.0,...,68.49,Yes,Auction.com,Exclude,Montgomery - TX,Exclude,Auction.com,Yes,NaN,NaN
73,Foreclosure,27022 CALAWAY FALLS LANE,KATY,Fort Bend,TX,77494.0,SALE_PENDING,2026-11-03,Single family home,4229.0,...,72.00,Yes,Auction.com,Exclude,Fort Bend - TX,Exclude,Lookup Needed,None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1936,Foreclosure,129 DUKE DR,FITZGERALD,Ben Hill,GA,31750,SALE_PENDING,2026-11-03,Single family home,2454.0,...,63.64,Yes,Auction.com,,Ben Hill - GA,Update,Auction.com,Yes,,NaN
1939,Foreclosure,920 BROOKMERE CT,ATLANTA,Fulton,GA,30349,SALE_PENDING,2026-11-03,Single family home,3636.0,...,73.97,Yes,Auction.com,Exclude,Fulton - GA,Exclude,Auction.com,None,NaN,NaN
1940,Foreclosure,254 VENTURE PATH,HIRAM,Paulding,GA,30141,SALE_PENDING,2026-11-03,Single family home,1562.0,...,61.54,Yes,Auction.com,Exclude,Paulding - GA,Exclude,Auction.com,None,NaN,NaN
1944,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Yes,Auction.com,,NaN,Add New,NaN,NaN,,NaN


In [ ]:
auction[['oldCounty', 'County', 'Market', 'Lead Manager Name']].head()

,oldCounty,County,Market,Lead Manager Name
0,Dallas,Dallas - TX,Dallas,Ernie Loreto
1,El Paso,El Paso - TX,TX Small County,Kevin Rapport
2,Harris,Harris - TX,Houston,Michael Shiner
3,Tarrant,Tarrant - TX,Dallas,Ernie Loreto
4,Harris,Harris - TX,Houston,Michael Shiner


In [ ]:
display(auction.loc[
    auction['Market'] == 'Exclude',
    ['Last Name', 'State', 'County', 'Market', 'Lead Manager Name', 'Decision']
])

,Last Name,State,County,Market,Lead Manager Name,Decision
8,"7031 Krause Dr, Missouri City, Tx 77489",TX,Fort Bend - TX,Exclude,NaN,Exclude
32,"14869 Prickly Pear Cove, Magnolia, Tx 77357",TX,Montgomery - TX,Exclude,NaN,Exclude
68,"22355 Dominguez Drive, Porter, Tx 77365",TX,Montgomery - TX,Exclude,NaN,Exclude
73,"27022 Calaway Falls Lane, Katy, Tx 77494",TX,Fort Bend - TX,Exclude,NaN,Exclude
81,"13736 Bighorn Trail, Willis, Tx 77378",TX,Montgomery - TX,Exclude,NaN,Exclude
...,...,...,...,...,...,...
1931,"673 Corsica Ln, Stockbridge, Ga 30281",GA,Henry - GA,Exclude,NaN,Exclude
1932,"145 Firewood St, Athens, Ga 30605",GA,Clarke - GA,Exclude,NaN,Exclude
1935,"6764 Columbus Rd, Lizella, Ga 31052",GA,Bibb - GA,Exclude,NaN,Exclude
1939,"920 Brookmere Ct, Atlanta, Ga 30349",GA,Fulton - GA,Exclude,NaN,Exclude


In [ ]:
# Remove duplicate properties based on Last Name
auction = auction.drop_duplicates(
    subset=['Last Name'],
    keep='first'
).reset_index(drop=True)

In [ ]:
before = len(auction)

auction = auction.drop_duplicates(
    subset=['Last Name'],
    keep='first'
).reset_index(drop=True)

print(f"Duplicates removed: {before - len(auction)}")
print(f"Remaining properties: {len(auction)}")

Duplicates removed: 0
Remaining properties: 1935


## Split Data

In [ ]:
export_data = Path('export_data')

# Export records marked Update
auction[auction['Decision'] == 'Update'].to_csv(
    export_data / 'update_auction.csv',
    index=False
)

# Export records marked Add New
auction[auction['Decision'] == 'Add New'].to_csv(
    export_data / 'create_auction.csv',
    index=False
)

# Confirm exported records
print("Update Auction:", (auction['Decision'] == 'Update').sum())
print("Create Auction:", (auction['Decision'] == 'Add New').sum())
print("Excluded:", (auction['Decision'] == 'Exclude').sum())

Update Auction: 1564
Create Auction: 1
Excluded: 370


## Format update_auction

In [ ]:
update_auction = auction[auction['Decision'] == 'Update'].copy()

#Add OB from Starting Bid
update_auction['Auction Opening Bid'] = (
    update_auction['Starting Bid'].replace(r'^\s*$', pd.NA, regex=True)
    .fillna('')
)

In [ ]:
# Get Auction Status from Salesforce
sf_status_lookup = sf_leads.set_index('Last Name')['Auction Status']

update_auction['Salesforce Auction Status'] = (
    update_auction['Salesforce Last Name'].map(sf_status_lookup)
)

Priority,	Condition,	                                Auction Status

1	        Original Status = CANCELED,	                Cancelled

2	        Salesforce Auction Status = Killed,	        Killed

3	        Salesforce Auction Status = Clear to Bid,	Clear to Bid

4	        Auction Opening Bid has a value,	         Active

5	        Everything else,	                         Pending

In [ ]:
# Default status
update_auction['Auction Status'] = 'Pending'

# Active if Opening Bid exists
has_bid = (
    update_auction['Auction Opening Bid'].notna() &
    update_auction['Auction Opening Bid'].astype(str).str.strip().ne('')
)

update_auction.loc[has_bid, 'Auction Status'] = 'Active'

# Preserve Salesforce Clear to Bid
#update_auction.loc[
    #update_auction['Salesforce Auction Status'] == 'Clear to Bid',
    #'Auction Status'
#] = 'Clear to Bid'

#Property type
update_auction.loc[
    update_auction['Property Type'] != 'Single family home',
    'Auction Status'
] = 'Killed'

# Canceled auction takes highest priority
update_auction.loc[
    update_auction['Status'] == 'CANCELED',
    'Auction Status'
] = 'Cancelled'

In [ ]:
display(update_auction[
    [
        'Last Name',
        'Decision',
        'Status',
        'Salesforce Auction Status',
        'Starting Bid',
        'Auction Opening Bid',
        'Auction Status'
    ]
].head(20))

print(update_auction['Auction Status'].value_counts())


,Last Name,Decision,Status,Salesforce Auction Status,Starting Bid,Auction Opening Bid,Auction Status
0,"1827 Poseidon Drive, Lancaster, Tx 75134",Update,SALE_PENDING,Cancelled,NaN,,Pending
1,"309 Adamtowne Dr, El Paso, Tx 79928",Update,SALE_PENDING,Killed,NaN,,Pending
2,"3011 Layne St, La Porte, Tx 77571",Update,SALE_PENDING,REO,NaN,,Pending
3,"7305 Hogan Dr, Benbrook, TX 76126",Update,SALE_PENDING,Cancelled,NaN,,Pending
4,"204 King William Dr, La Porte, Tx 77571",Update,SALE_PENDING,Killed,NaN,,Pending
5,"2732 Trophy Club Dr, Trophy Club, Tx 76262",Update,SALE_PENDING,Cancelled,NaN,,Pending
6,"20539 Avery Grove Ct, Cypress, Tx 77433",Update,SALE_PENDING,Cancelled,NaN,,Pending
7,"801 N D St, Midland, Tx 79701",Update,SALE_PENDING,Cancelled,NaN,,Pending
9,"507 Longworth Blvd, Duncanville, Tx 75116",Update,SALE_PENDING,Pending,NaN,,Pending
10,"5913 Lake Hubbard Pkwy Bldg: B Suite: 121, Gar...",Update,SALE_PENDING,Killed,NaN,,Killed


Auction Status
Pending      1141
Cancelled     305
Killed         69
Active         49
Name: count, dtype: int64


In [ ]:
# Add Killed Reason
update_auction['Killed Reason'] = update_auction['Auction Status'].apply(
    lambda x: 'Property Type' if x == 'Killed' else ''
)

In [ ]:
display(update_auction.loc[
    (update_auction['Auction Status'] == 'Killed') &
    (update_auction['Property Type'] != 'Single family home')
])

,Listing Type,Property Address,Property City,oldCounty,State,Zip,Status,Auction Date,Property Type,Square Footage,...,County,Decision,Salesforce Auction Trustee,Salesforce Online Bidding,Lead Manager Name,Lead Manager,Auction Opening Bid,Salesforce Auction Status,Auction Status,Killed Reason
10,Foreclosure,5913 LAKE HUBBARD PKWY 121,GARLAND,Dallas,TX,75043.0,SALE_PENDING,2026-11-03,Condominium unit,920.0,...,Dallas - TX,Update,Lookup Needed,None,Ernie Loreto,005Qg00000UWrpV,,Killed,Killed,Property Type
14,Foreclosure,2910 ACOPIO BEND UNIT 61,AUSTIN,Travis,TX,78745.0,SALE_PENDING,2026-11-03,Condominium unit,1458.0,...,Travis - TX,Update,Auction.com,Yes,Aaron Amuchastegui,005Qg00000Voakn,,Killed,Killed,Property Type
40,Foreclosure,4229 LUGARD DR,KILLEEN,Bell,TX,76542.0,SALE_PENDING,2026-11-03,Land,0.0,...,Bell - TX,Update,Auction.com,Yes,Evan Carr,005Qg00000V4NiY,,Killed,Killed,Property Type
42,Foreclosure,20730 DAPPLED RIDGE WAY,HUMBLE,Harris,TX,77338.0,SALE_PENDING,2026-11-03,Single family attached,4414.0,...,Harris - TX,Update,Xome.com,Yes,Michael Shiner,005Qg00000Nsy1x,,Killed,Killed,Property Type
61,Foreclosure,5025 MISSION AVE,DALLAS,Dallas,TX,75206.0,SALE_PENDING,2026-11-03,Duplex building,2814.0,...,Dallas - TX,Update,Lookup Needed,No,Ernie Loreto,005Qg00000UWrpV,,Cancelled,Killed,Property Type
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1740,Foreclosure,1314 RUTH STREET,GRIFFIN,Spalding,GA,30223,SALE_PENDING,2026-11-03,Duplex building,1675.0,...,Spalding - GA,Update,Auction.com,None,David Riehle,0051H00000ASDES,,Cancelled,Killed,Property Type
1773,Foreclosure,909 HIGHWAY 314,FAYETTEVILLE,Fayette,GA,30214,SALE_PENDING,2026-11-03,Unknown,NaN,...,Fayette - GA,Update,Auction.com,None,David Riehle,0051H00000ASDES,,Pending,Killed,Property Type
1816,Foreclosure,510 ROCK QUARRY DRIVE,DANIELSVILLE,Madison,GA,30633,SALE_PENDING,2026-11-03,Commercial,2080.0,...,Madison - GA,Update,Auction.com,None,,NaN,,Cancelled,Killed,Property Type
1867,Foreclosure,1965 NOCTURNE DR,ALPHARETTA,FULTON,GA,30009,SALE_PENDING,2026-11-03,Condominium unit,1209.0,...,FULTON - GA,Update,Auction.com,None,,NaN,,Killed,Killed,Property Type


In [ ]:
#FIRST IMPORT ONLY

#Format Auction Date Updated as MM/DD/YYYY
update_auction['Auction Date Updated'] = (
    pd.to_datetime(update_auction['Auction Date'], errors='coerce')
    .dt.strftime('%m/%d/%Y')
    .fillna('')
)

update_auction = update_auction.rename(columns={
    'Status': 'notStatus',
    'Year Built': 'notYearBuilt',
    'Baths': 'notBaths',
    'Trustee Number': 'NS Trustee Sale Number',
    'Market': 'notMarket',
    'County':'notCounty'
})

In [ ]:
#ADDITIONAL IMPORTS
#Rename columns

#update_auction = update_auction.rename(columns={
    #'Status': 'notStatus',
    #'Year Built': 'notYearBuilt',
    #'Baths': 'notBaths',
    #'Trustee Number': 'NS Trustee Sale Number',
    #'Market': 'notMarket',
    #'County':'notCounty'
    #'Lead Manager': 'notLeadManager'
#})

In [ ]:
update_auction.to_csv(
    export_data / 'update_auction.csv',
    index=False
)

## Format create_auction

In [ ]:
create_auction = auction[auction['Decision'] == 'Add New'].copy()

In [ ]:
#Format Tax ID

create_auction['Tax ID'] = (
    create_auction['APN']
    .astype('string')
    .replace(r'(?i)^\s*(null|nan|none)?\s*$', '', regex=True)
    .fillna('')
)

create_auction = create_auction.rename(columns={
    'APN': 'notAPN'
})

In [ ]:
#Create unique identifier

market_map = {
    'TX Small County': 'TXSMALLCOUNTY',
    'Austin': 'AUSTIN',
    'Waco': 'WACO',
    'Dallas': 'DAL',
    'San Antonio': 'SAN_ANT',
    'Houston': 'HOUS'
}

create_auction['APN'] = (
    create_auction['Market'].map(market_map)
    + '_' + create_auction['Property Address'].astype('string').str.strip().str.replace(r'\s+', '_', regex=True)
    + '_' + create_auction['Zip'].astype('string').str.replace(r'\.0$', '', regex=True)
)

In [ ]:
#Type of Property
create_auction['Type of Property'] = (
    create_auction['Property Type'].replace({
        'Single family home': 'SFR'
    })
)

#Auction Opening Bid
create_auction['Auction Opening Bid'] = (
    create_auction['Starting Bid']
    .replace(r'^\s*$', pd.NA, regex=True)
    .fillna('')
)

#Auction Status
create_auction['Auction Status'] = 'Pending'

# Active if opening bid exists
has_bid = (
    create_auction['Auction Opening Bid'].notna() &
    create_auction['Auction Opening Bid'].astype(str).str.strip().ne('')
)

create_auction.loc[
    has_bid,
    'Auction Status'
] = 'Active'

# Killed if not SFR (highest priority)
create_auction.loc[
    create_auction['Type of Property'] != 'SFR',
    'Auction Status'
] = 'Killed'

# Add Killed Reason
create_auction['Killed Reason'] = create_auction['Auction Status'].apply(
    lambda x: 'Property Type' if x == 'Killed' else ''
)

#Auction Date Updated - MM/DD/YYYY
create_auction['Auction Date Updated'] = (
    pd.to_datetime(create_auction['Auction Date'], errors='coerce')
    .dt.strftime('%m/%d/%Y')
    .fillna('')
)

#Rename columns
create_auction = create_auction.rename(columns={
    'Est. Market Value': 'RPR',
    'Bedrooms': 'Beds',
    'Trustee Number': 'NS Trustee Sale Number',
    ' Square Footage': 'Bldg Sq Ft',
    'Status': 'notStatus'
})

/var/folders/0v/2p48l9mx23dc0422gf24z_qr0000gn/T/ipykernel_51608/656796701.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  create_auction['Property Type'].replace({


In [ ]:
# Remove trailing .0 from numeric columns
for col in ['Auction Opening Bid', 'RPR', 'Beds', 'Bldg Sq Ft', 'Baths']:
    create_auction[col] = (
        create_auction[col]
        .astype('string')
        .str.replace(r'\.0$', '', regex=True)
        .fillna('')
    )

In [ ]:
display(create_auction[
    [
        'Last Name',
        'Market',
        'County',
        'State',
        'APN',
        'Tax ID',
        'notAPN',
        'Type of Property',
        'Auction Opening Bid',
        'Auction Status',
        'Auction Date Updated',
        'RPR',
        'Beds',
        'Baths',
        'NS Trustee Sale Number',
        'Bldg Sq Ft',
        'notStatus'
    ]
].head(20))

print(create_auction['Auction Status'].value_counts())

,Last Name,Market,County,State,APN,Tax ID,notAPN,Type of Property,Auction Opening Bid,Auction Status,Auction Date Updated,RPR,Beds,Baths,NS Trustee Sale Number,Bldg Sq Ft,notStatus
1180,NaN,,NaN,NaN,<NA>,,NaN,NaN,,Killed,,,,,NaN,,NaN


Auction Status
Killed    1
Name: count, dtype: int64


In [ ]:
create_auction.head(5)


,Listing Type,Property Address,Property City,oldCounty,State,Zip,notStatus,Auction Date,Property Type,Bldg Sq Ft,...,Salesforce Online Bidding,Lead Manager Name,Lead Manager,Tax ID,APN,Type of Property,Auction Opening Bid,Auction Status,Killed Reason,Auction Date Updated
1180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,...,NaN,,NaN,,<NA>,NaN,,Killed,Property Type,


In [ ]:
create_auction.to_csv(
    export_data / 'create_auction.csv',
    index=False
)